# 03 · Retrieval: vector vs BM25 vs hybrid

Where does each retrieval arm win, and why does fusing them help?

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
from src.retrieval.hybrid_search import make_retriever
from src.retrieval.index import RagIndex
from src.retrieval.reranker import get_reranker

index = RagIndex.load("default")
print(index.manifest)

## A query where lexical search wins\n\nExact drug names are rare tokens: BM25 rewards them through IDF, while a bi-encoder maps them into a fuzzy neighbourhood.

In [ ]:
def compare(query, k=5):
    rows = []
    for strategy in ["vector", "bm25", "hybrid"]:
        for hit in make_retriever(strategy, index, top_k=k).retrieve(query):
            rows.append({
                "strategy": strategy,
                "rank": hit.rank,
                "score": round(hit.score, 4),
                "title": hit.chunk.title[:58],
                "page": hit.chunk.page,
                "section": hit.chunk.section[:28],
            })
    return pd.DataFrame(rows)

compare("What are the most common adverse reactions to amlodipine?")

## A query where semantic search wins\n\nParaphrased clinical intent shares few literal tokens with the source text.

In [ ]:
compare("How do drugs that block the renin angiotensin system lower blood pressure?")

## Overlap between the arms\n\nIf the two arms returned the same documents, fusion would add nothing.

In [ ]:
queries = [
    "first-line treatments for hypertension",
    "adverse reactions to amlodipine",
    "mechanism of action of SGLT2 inhibitors",
    "anticoagulation for atrial fibrillation",
    "risk factors for chronic kidney disease",
    "management of sepsis in the emergency department",
]

overlaps = []
for q in queries:
    v = {h.chunk.chunk_id for h in make_retriever("vector", index, top_k=20).retrieve(q)}
    b = {h.chunk.chunk_id for h in make_retriever("bm25", index, top_k=20).retrieve(q)}
    overlaps.append({
        "query": q[:44],
        "vector_only": len(v - b),
        "shared": len(v & b),
        "bm25_only": len(b - v),
        "jaccard": round(len(v & b) / max(len(v | b), 1), 3),
    })

overlaps = pd.DataFrame(overlaps)
overlaps.plot.barh(x="query", stacked=True, y=["vector_only", "shared", "bm25_only"],
                   color=["#4C9BE8", "#9E9E9E", "#5BB98C"], title="Top-20 overlap")
plt.tight_layout()
overlaps

Low Jaccard overlap is exactly why hybrid retrieval helps: the two arms surface substantially different candidates.

## What reranking changes

In [ ]:
query = "What are the recommended first-line treatments for hypertension?"
candidates = make_retriever("hybrid", index, top_k=20).retrieve(query)
reranked = get_reranker().rerank(query, list(candidates), top_n=5)

before = {c.chunk.chunk_id: c.rank for c in candidates}
pd.DataFrame([{
    "new_rank": c.rank,
    "old_rank": before.get(c.chunk.chunk_id),
    "moved": before.get(c.chunk.chunk_id, 0) - c.rank,
    "rerank_score": round(c.rerank_score, 3),
    "title": c.chunk.title[:56],
    "section": c.chunk.section[:26],
} for c in reranked])

Positive `moved` means the cross-encoder promoted a passage the bi-encoder had ranked lower. Note also the per-document quota: no single paper is allowed to occupy every slot.